In [1]:
# 常用的reducer函数
# add, 集合的add，做并集
# add_messages: 有具体的数据类型，并且是langgraph，专门用来何必跟消息列表的reducer函数，  维护对话历史状态字段，记录不同的模型调用的时候， 用户传的提示词，模型返回的历史数据， 当作对话历史，
# 返回的数据类型是Messages， Langgraph自带的，应该使用具体的子类，add 合并的时候，不是普通的字符串列表拼接
# [hello, hi] + [hello] =普通python =》 [hello, hi, hello]
# add_message:  包含id属性，根据id合并， 如果right新信息id不存在则追加，如果right id存在，会用新的替换旧的覆盖

In [2]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages

left = [
    SystemMessage(content="你是一个专业的翻译", id = '1'), # additional_kwargs={}, response_metadata={}这些参数没有用到
    HumanMessage(content="你好", id = '2'), # 被覆盖了，后面是新的
    AIMessage(content="你好，我是专业的翻译", id = '3') # 被覆盖
]

right = [
    HumanMessage(content="我是老王，你是小王", id='2'),
    AIMessage(content="好的我记住了", id='3'),
    HumanMessage(content="你是谁", id='4'),
    AIMessage(content="我是小王", id='5')
]

merged = add_messages(left, right)
print(merged)

[SystemMessage(content='你是一个专业的翻译', additional_kwargs={}, response_metadata={}, id='1'), HumanMessage(content='我是老王，你是小王', additional_kwargs={}, response_metadata={}, id='2'), AIMessage(content='好的我记住了', additional_kwargs={}, response_metadata={}, id='3', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你是谁', additional_kwargs={}, response_metadata={}, id='4'), AIMessage(content='我是小王', additional_kwargs={}, response_metadata={}, id='5', tool_calls=[], invalid_tool_calls=[])]


In [3]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages

left = [
    SystemMessage(content="你是一个专业的翻译", id = '1'), # additional_kwargs={}, response_metadata={}这些参数没有用到
    HumanMessage(content="你好", id = '2'), # 被覆盖了，后面是新的
    AIMessage(content="你好，我是专业的翻译", id = '3') # 被覆盖
]

right = [
    HumanMessage(content="我是老王，你是小王", id='2'),
    AIMessage(content="好的我记住了", id='3'),
    HumanMessage(content="你是谁", id='4'),
    AIMessage(content="你是小王吗？", id='1') # 不会因为AImessage和SystemMessage不同判断不同信息，只会根据id来
]

merged = add_messages(left, right)
print(merged)

[AIMessage(content='你是小王吗？', additional_kwargs={}, response_metadata={}, id='1', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='我是老王，你是小王', additional_kwargs={}, response_metadata={}, id='2'), AIMessage(content='好的我记住了', additional_kwargs={}, response_metadata={}, id='3', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你是谁', additional_kwargs={}, response_metadata={}, id='4')]


In [4]:
# 默认行为
# 在状态里面的属性值，没有显式定义reducer，  没有使用Annotation去添加reucer，会导致默认的reducer机制，就是覆盖，不断的覆盖，不能追加合并和累加
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class OverAllState(TypedDict):
    logs: list[str]
    id: str

def node_a(state: OverAllState):
    return {
        "logs": ["node_a"],
        "id": "node_a"
    }

def node_b(state: OverAllState):
    return {
        "logs": ["node_b"], # 直接覆盖，后面覆盖前面的， 所以需要定义reducer
        "id": "node_b" # 直接覆盖
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)

graph = builder.compile()
result = graph.invoke({"logs": ["START"], "id": "start"})
print('=' * 30, '-> result <-', '=' * 30)
print(result)

============================== -> result <- ==============================
{'logs': ['node_b'], 'id': 'node_b'}
